[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_04_data_science_math/ch17_pandas_foundations/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 17 Walkthrough: Pandas Foundations — Series, DataFrames, Indexing, Filtering, and Reshaping

**Book Reference:** Chapter 17, Sections 17.1 – 17.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every mutation-free pattern here follows modern **Pandas 3.x Copy-on-Write** best practices.

In [ ]:
# @title Step 0: Click Play (▶) to Load Pandas & the Bundled Clinic Triage Dataset
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import pandas as pd
import sharukh_tutor as tutor

df = tutor.load_data("clinic_triage", as_pandas=True)
print("Loaded clinic_triage with shape:", df.shape)


## Section 17.3.1 — `Series` vs. `DataFrame` Anatomy & Quick Summaries

In [ ]:
print("Shape:", df.shape)
print("DataFrame type:", type(df).__name__)
print("Series type:   ", type(df["temp_c"]).__name__)
print("Mean temp_c:   ", round(float(df["temp_c"].mean()), 2))
print("Median temp_c: ", round(float(df["temp_c"].median()), 2))
print("Std temp_c:    ", round(float(df["temp_c"].std()), 2))


## Section 17.3.2 — Label vs. Position Indexing (`.loc` vs. `.iloc`)

In [ ]:
print("iloc[0:3, 0:3] (exclusive stop):")
print(df.iloc[0:3, 0:3])

by_id = df.set_index("patient_id")
print("\nloc['UHC-0001':'UHC-0003', ['district', 'temp_c']] (inclusive):")
print(by_id.loc["UHC-0001":"UHC-0003", ["district", "temp_c"]])


## Section 17.3.3 — Boolean Masking with `&`, `|`, `~`, and `.isin()`

In [ ]:
adult_fever = (df["age_years"] >= 18) & (df["temp_c"] >= 38.0)
target_dist = df["district"].isin(["Dhaka", "Sylhet"])

sub = df.loc[adult_fever & target_dist].reset_index(drop=True)
other_count = int((~target_dist).sum())

print("Adult fever rows (all districts):", int(adult_fever.sum()))
print("Adult fever in Dhaka/Sylhet:     ", len(sub))
print("Outside Dhaka/Sylhet rows:       ", other_count)
print(sub[["patient_id", "district", "age_years", "temp_c"]].head(3))


## Section 17.3.4 — Vectorized Columns, `.assign()`, & Pandas 3.x Copy-on-Write

In [ ]:
enriched = df.assign(
    bmi=(df["weight_kg"] / (df["height_cm"] / 100) ** 2).round(2),
    is_anemic=df["hemoglobin_g_dl"] < 11.0,
)

print("'bmi' in original df?", "bmi" in df.columns)
print("'bmi' in enriched?   ", "bmi" in enriched.columns)
print("Anemic patients:     ", int(enriched["is_anemic"].sum()))
cols = ["patient_id", "bmi", "hemoglobin_g_dl", "is_anemic"]
print(enriched[cols].head(3))


## Section 17.3.5 — Deterministic Multi-Key Sorting with `.sort_values()`

In [ ]:
cols = ["patient_id", "district", "temp_c", "triage_priority"]
top_fever = (
    df.sort_values(
        by=["temp_c", "patient_id"],
        ascending=[False, True],
    )[cols]
    .head(5)
    .reset_index(drop=True)
)
print(top_fever)


## Section 17.3.6 — Reshaping Data: `pd.melt`, `pd.crosstab`, & `pivot_table`

In [ ]:
wide_bp = pd.DataFrame(
    {
        "patient_id": ["UHC-0001", "UHC-0002", "UHC-0003"],
        "sbp_day1": [123, 132, 140],
        "sbp_day2": [119, 128, 134],
    }
)
long_bp = pd.melt(
    wide_bp,
    id_vars=["patient_id"],
    value_vars=["sbp_day1", "sbp_day2"],
    var_name="visit",
    value_name="sbp",
)
print("Long BP shape:", long_bp.shape)
print(long_bp)

ct = pd.crosstab(df["district"], df["triage_priority"])
print("\nDistrict x Triage Priority counts:")
print(ct)

mean_temp = df.pivot_table(
    index="district",
    columns="triage_priority",
    values="temp_c",
    aggfunc="mean",
).round(2)
print("\nMean Temp (C) by District x Priority:")
print(mean_temp)


## Section 17.3.7 — Break It On Purpose! (Pandas Error Anatomy)

In [ ]:
# Intentional Bug #1: KeyError — nonexistent column name
print(df["hemoglobin"].mean())


In [ ]:
# Intentional Bug #2: ValueError — using Python `and` on Series
bad_mask = (df["age_years"] >= 18) and (df["temp_c"] >= 38.0)
print(bad_mask.sum())


In [ ]:
# Intentional Bug #3: IndexError — .iloc out-of-bounds row position
print(df.iloc[120])


## Section 17.4 — Stage 4 Blueprint Implementation: District Fever-Rate Audit

In [ ]:
cohort = (
    df.loc[df["age_years"] >= 18]
    .copy()
    .assign(
        bmi=lambda d: (
            d["weight_kg"] / (d["height_cm"] / 100) ** 2
        ).round(2),
        has_fever=lambda d: d["temp_c"] >= 38.0,
    )
    .reset_index(drop=True)
)

fever_table = pd.crosstab(cohort["district"], cohort["has_fever"])
fever_table.columns = ["afebrile", "febrile"]
total_adults = fever_table["afebrile"] + fever_table["febrile"]
fever_table["fever_pct"] = (
    fever_table["febrile"] / total_adults * 100
).round(1)
ranked = fever_table.sort_values("fever_pct", ascending=False)

print(
    "Adult cohort size:", len(cohort),
    "| Original df cols:", len(df.columns),
)
print(ranked)
